In [3]:
!pip install scikit-learn numpy matplotlib pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl (10.0 MB)

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
data = pd.read_csv('train.csv')
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
data = np.array(data)
np.random.shuffle(data)

val_size = 1000
val_data = data[:val_size]
train_data = data[val_size:]

Y_val = val_data[:, 0]
x_val = val_data[:, 1:] / 255.0

Y_train = train_data[:, 0]
x_train = train_data[:, 1:] / 255.0

In [4]:
# test.csv has no label column (784 pixel columns only) — it's unlabeled Kaggle
# submission data, not usable for accuracy evaluation. Use x_val/Y_val for that.
test = pd.read_csv('test.csv')
x_test = np.array(test) / 255.0

In [5]:
def init_params():
    # np.random.rand alone returns only positive values in [0, 1); with 784
    # summed inputs that makes the first forward pass wildly overconfident,
    # triggers a huge first gradient step, and kills most ReLU units for good.
    # Centering to [-0.5, 0.5) keeps the initial activations sane so training
    # actually converges.
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def relu(x):
    return np.maximum(0, x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=0, keepdims=True))
    return exp_x / np.sum(exp_x, axis=0, keepdims=True)

def one_hot_encode(y):
    one_hot_Y= np.zeros((y.size, y.max()+1))
    one_hot_Y[np.arange(y.size), y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def forward_prop(W1, b1, W2, b2, x):
    Z1=W1.dot(x) + b1
    A1=relu(Z1)
    Z2=W2.dot(A1) + b2
    A2=softmax(Z2)
    return Z1, A1, Z2, A2

def back_prop(W1, b1, W2, b2, Z1, A1, Z2, A2, x, y):
    one_hot_Y = one_hot_encode(y)
    m = y.size
    dZ2 = A2 - one_hot_Y
    dW2 = (1/m) * dZ2.dot(A1.T) #this here does
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * (Z1 > 0)
    dW1 = (1/m) * dZ1.dot(x.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate):
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    return W1, b1, W2, b2


In [6]:
def get_predictions(A2):
    return np.argmax(A2, axis=0)

def get_accuracy(predictions, y):
    return np.mean(predictions == y) * 100

def gradient_descent(x, y, learning_rate=0.01, iterations=100):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, x)
        dW1, db1, dW2, db2 = back_prop(W1, b1, W2, b2, Z1, A1, Z2, A2, x, y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)
        if i % 10 == 0:
            print("Iterations:", i)
            print("Accuracy: ", get_accuracy(get_predictions(A2), y))
    return W1, b1, W2, b2

In [7]:
W1, b1, W2, b2 = gradient_descent(x_train.T, Y_train, learning_rate=0.5, iterations=900)

Iterations: 0
Accuracy:  11.582926829268294
Iterations: 10
Accuracy:  42.009756097560974
Iterations: 20
Accuracy:  57.08780487804878
Iterations: 30
Accuracy:  66.5219512195122
Iterations: 40
Accuracy:  68.43658536585366
Iterations: 50
Accuracy:  72.9560975609756
Iterations: 60
Accuracy:  77.3829268292683
Iterations: 70
Accuracy:  79.80731707317074
Iterations: 80
Accuracy:  81.33170731707317
Iterations: 90
Accuracy:  82.46829268292683
Iterations: 100
Accuracy:  83.53170731707317
Iterations: 110
Accuracy:  84.42439024390244
Iterations: 120
Accuracy:  85.17560975609756
Iterations: 130
Accuracy:  85.76341463414634
Iterations: 140
Accuracy:  86.30975609756098
Iterations: 150
Accuracy:  86.84878048780487
Iterations: 160
Accuracy:  87.25853658536586
Iterations: 170
Accuracy:  87.53170731707317
Iterations: 180
Accuracy:  87.81219512195122
Iterations: 190
Accuracy:  88.09512195121951
Iterations: 200
Accuracy:  88.31219512195122
Iterations: 210
Accuracy:  88.5170731707317
Iterations: 220
Accurac

In [9]:
_, _, _, A2_val = forward_prop(W1, b1, W2, b2, x_val.T)
val_predictions = get_predictions(A2_val)
print("Validation accuracy:", get_accuracy(val_predictions, Y_val))

Validation accuracy: 91.7


In [10]:
np.savetxt('W1.txt', W1)
np.savetxt('b1.txt', b1)
np.savetxt('W2.txt', W2)
np.savetxt('b2.txt', b2)
print("Saved weights to W1.txt, b1.txt, W2.txt, b2.txt")

Saved weights to W1.txt, b1.txt, W2.txt, b2.txt
